# V18 Supplementary Table S1/S2 Generation
**Run AFTER the verification notebook (Cells 1-7).** Variables , , , , , , ,  must be in memory.

Cells 8-11 generate and verify S1/S2 xlsx files, saved directly to Google Drive.

In [ ]:
# =================================================================
# CELL 8: INSTALL OPENPYXL + DEFINE STYLES
# =================================================================
!pip install openpyxl -q

import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

OUT_DIR = '/content/drive/MyDrive/ITLAS/results/version18-analysis'

# Styles
header_font = Font(name='Arial', bold=True, size=10, color='FFFFFF')
header_fill = PatternFill('solid', fgColor='2F5496')
cat_fill = PatternFill('solid', fgColor='D6E4F0')
data_font = Font(name='Arial', size=10)
title_font = Font(name='Arial', bold=True, size=12)
thin_border = Border(
    left=Side(style='thin', color='CCCCCC'),
    right=Side(style='thin', color='CCCCCC'),
    top=Side(style='thin', color='CCCCCC'),
    bottom=Side(style='thin', color='CCCCCC')
)
green_fill = PatternFill('solid', fgColor='E2EFDA')
orange_fill = PatternFill('solid', fgColor='FCE4D6')

def style_header(ws, row, ncols):
    for c in range(1, ncols+1):
        cell = ws.cell(row=row, column=c)
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        cell.border = thin_border

def style_data(ws, row, ncols, fill=None):
    for c in range(1, ncols+1):
        cell = ws.cell(row=row, column=c)
        cell.font = data_font
        cell.border = thin_border
        if fill:
            cell.fill = fill

# Verify prerequisites
c3_set = set(C3_GENES)
c5_set = set(C5_GENES)
shared = c3_set & c5_set
print(f"Prerequisites check:")
print(f"  C3: {len(c3_set)}, C5: {len(c5_set)}, Shared: {len(shared)}")
print(f"  Pathways: {len(ALL_29)}")
print(f"  Unique pathway genes: {len(all_pw_genes_29)}")
print(f"✅ Cell 8 ready")

In [ ]:
# =================================================================
# CELL 9: GENERATE TABLE S1 (29 Pathway Definitions)
# =================================================================
print("=" * 70)
print("  GENERATING SUPPLEMENTARY TABLE S1")
print("=" * 70)

CATEGORIES = {
    'inflammasome': 'Immune Effector', 'cytotoxicity': 'Immune Effector',
    'checkpoint': 'Immune Effector', 'exhaustion': 'Immune Effector',
    'nk_function': 'Immune Effector',
    'nk_il15_dual': 'Immune Regulation', 'il15_mtor': 'Immune Regulation',
    'immune_evasion': 'Immune Regulation', 'treg': 'Immune Regulation',
    'naive_t': 'Immune Regulation',
    'memory_t': 'T Cell Biology', 'tf_programs': 'T Cell Biology',
    'tissue_resident': 'T Cell Biology', 'stemness': 'T Cell Biology',
    'glycolysis': 'Metabolism', 'oxphos': 'Metabolism',
    'mito_dysfunction': 'Metabolism', 'metabolic_recovery': 'Metabolism',
    'cancer_associated': 'Disease Progression', 'fibrosis': 'Disease Progression',
    'senescence': 'Disease Progression', 'epigenetics': 'Disease Progression',
    'angiogenesis': 'Disease Progression', 'cell_cycle': 'Disease Progression',
    'proliferation': 'Disease Progression', 'apoptosis': 'Disease Progression',
    'antigen_presentation': 'Antigen Processing', 'type1_ifn': 'Interferon Response',
    'tgfb_signaling': 'TGF-beta Signaling',
}

pathway_order = [
    'inflammasome', 'cytotoxicity', 'checkpoint', 'exhaustion', 'nk_function',
    'nk_il15_dual', 'il15_mtor', 'immune_evasion', 'treg', 'naive_t',
    'memory_t', 'tf_programs', 'tissue_resident', 'stemness',
    'glycolysis', 'oxphos', 'mito_dysfunction', 'metabolic_recovery',
    'cancer_associated', 'fibrosis', 'senescence', 'epigenetics',
    'angiogenesis', 'cell_cycle', 'proliferation', 'apoptosis',
    'antigen_presentation', 'type1_ifn', 'tgfb_signaling'
]

# Count genes found in h5ad per pathway
try:
    h5ad_genes_set = set(adata.var_names)
except:
    h5ad_genes_set = None
    print("  ⚠️ h5ad not in memory, skipping h5ad check column")

wb1 = openpyxl.Workbook()
ws1 = wb1.active
ws1.title = "S1_Pathway_Definitions"

# Title
ws1.cell(row=1, column=1, value="Supplementary Table S1. Gene Set Definitions for 29 Immunological Pathways Used in AUCell Pathway Scoring (C2/C4)")
ws1.cell(row=1, column=1).font = title_font
ws1.merge_cells('A1:G1')

n_unique = len(all_pw_genes_29)
n_found = len(all_found_genes) if 'all_found_genes' in dir() else n_unique
n_missing_genes = len(all_missing_genes) if 'all_missing_genes' in dir() else 0
ws1.cell(row=2, column=1, value=f"ITLAS v18 tissue-separated reanalysis of GSE182159. 26 original pathways (C2) + 3 added in C9 (marked *). {n_unique} unique genes defined; {n_found} found in h5ad ({n_missing_genes} absent: {', '.join(sorted(all_missing_genes)) if 'all_missing_genes' in dir() and all_missing_genes else 'none'}).")
ws1.cell(row=2, column=1).font = Font(name='Arial', size=9, italic=True)
ws1.merge_cells('A2:G2')

# Headers
headers = ['Category', 'Pathway Name', 'N Genes (Defined)', 'N Genes (in h5ad)', 'Gene List', 'Source', 'Shared with Other Pathways']
for i, h in enumerate(headers, 1):
    ws1.cell(row=4, column=i, value=h)
style_header(ws1, 4, 7)

row = 5
current_cat = None
for pw in pathway_order:
    genes = ALL_29[pw]
    cat = CATEGORIES[pw]
    is_new = pw in GENE_SETS_3NEW

    fill = None
    if cat != current_cat:
        current_cat = cat
        fill = cat_fill

    pw_display = f"{pw}*" if is_new else pw
    source = "C9 (added)" if is_new else "C2 (original)"

    # h5ad availability
    if h5ad_genes_set:
        found = [g for g in genes if g in h5ad_genes_set]
        missing = [g for g in genes if g not in h5ad_genes_set]
        n_found_pw = len(found)
        gene_display = ', '.join(found)
        if missing:
            gene_display += f' [ABSENT: {", ".join(missing)}]'
    else:
        n_found_pw = len(genes)
        gene_display = ', '.join(genes)

    # Shared genes
    shared_with = []
    for other_pw, other_genes in ALL_29.items():
        if other_pw != pw:
            overlap = set(genes) & set(other_genes)
            if overlap:
                for g in overlap:
                    shared_with.append(f"{g}({other_pw})")
    shared_str = '; '.join(sorted(set(shared_with)))[:250] if shared_with else ""

    ws1.cell(row=row, column=1, value=cat)
    ws1.cell(row=row, column=2, value=pw_display)
    ws1.cell(row=row, column=3, value=len(genes))
    ws1.cell(row=row, column=4, value=n_found_pw)
    ws1.cell(row=row, column=5, value=gene_display)
    ws1.cell(row=row, column=6, value=source)
    ws1.cell(row=row, column=7, value=shared_str)
    style_data(ws1, row, 7, fill=fill)
    ws1.cell(row=row, column=3).alignment = Alignment(horizontal='center')
    ws1.cell(row=row, column=4).alignment = Alignment(horizontal='center')
    row += 1

# Summary
row += 1
ws1.cell(row=row, column=1, value="TOTAL")
ws1.cell(row=row, column=2, value=f"{len(ALL_29)} pathways")
ws1.cell(row=row, column=3, value=n_unique)
ws1.cell(row=row, column=4, value=n_found)
ws1.cell(row=row, column=5, value=f"{n_unique} unique genes defined; {len(multi)} genes shared across ≥2 pathways")
for c in range(1, 8):
    ws1.cell(row=row, column=c).border = thin_border
    ws1.cell(row=row, column=c).font = Font(name='Arial', bold=True, size=10)

# Column widths
ws1.column_dimensions['A'].width = 20
ws1.column_dimensions['B'].width = 24
ws1.column_dimensions['C'].width = 14
ws1.column_dimensions['D'].width = 14
ws1.column_dimensions['E'].width = 80
ws1.column_dimensions['F'].width = 14
ws1.column_dimensions['G'].width = 60

s1_path = f'{OUT_DIR}/Supplementary_Table_S1.xlsx'
wb1.save(s1_path)
print(f"  ✅ S1 saved: {s1_path}")
print(f"     {len(ALL_29)} pathways, {n_unique} unique genes, {n_found} in h5ad")

In [ ]:
# =================================================================
# CELL 10: GENERATE TABLE S2 (196 Gene Candidates)
# =================================================================
print("=" * 70)
print("  GENERATING SUPPLEMENTARY TABLE S2")
print("=" * 70)

# C9B genes
C9B_GENES = [
    'MX1', 'ISG15', 'STAT2', 'IRF3', 'IRF7', 'SOCS1', 'SOCS3', 'AICDA', 'JCHAIN',
    'HLA-DRA', 'HLA-DRB1', 'HLA-DPB1', 'HLA-DPA1', 'CD74', 'B2M', 'TAP1',
    'IL1RN', 'STAT1', 'IL1B'
]

# Key manuscript roles
KEY_ROLES = {
    'MX1': 'IFN response, IT-specific (FDR Tier 1)',
    'ISG15': 'IFN response, IT-specific (FDR Tier 1)',
    'STAT2': 'IFN response, IT-specific (FDR Tier 1)',
    'IRF3': 'IFN response, IT-specific (FDR Tier 1)',
    'IRF7': 'IFN response (FDR Tier 1)',
    'SOCS1': 'JAK-STAT brake, IT-specific (FDR Tier 1)',
    'SOCS3': 'JAK-STAT brake, IT-specific (FDR Tier 1)',
    'DNMT1': 'Epigenetic silencing, pan-tissue (FDR Tier 1)',
    'DNMT3A': 'Epigenetic silencing (FDR Tier 1)',
    'TGFB1': 'Paracrine suppression (FDR Tier 1)',
    'LGALS9': 'Paracrine suppression (FDR Tier 1)',
    'MTOR': 'Metabolic checkpoint (FDR Tier 1)',
    'JAK1': 'Most frequent sig gene (32 tests)',
    'TOX': 'Liver-specific exhaustion (Tier 2)',
    'PRDM1': 'Differentiation block, pan-tissue (Tier 2)',
    'HLA-DRA': 'Antigen presentation (FDR Tier 1)',
    'HLA-DRB1': 'Antigen presentation (FDR Tier 1)',
    'HLA-DPB1': 'Antigen presentation (FDR Tier 1)',
    'HLA-DPA1': 'Antigen presentation (FDR Tier 1)',
    'CD74': 'Antigen presentation (FDR Tier 1)',
    'AICDA': 'B cell CSR (Tier 3, sparse)',
    'JCHAIN': 'B cell phenotype (FDR Tier 1)',
    'IL2RA': 'B cell activation',
    'TYROBP': 'PlasmaB cytotoxic phenotype',
    'TFAM': 'Mito biogenesis, pan-immune',
    'TGFBR2': 'Pan-immune, rho=0.88 with JAK1',
    'EIF4EBP1': 'Metabolic dissociation (Liver NK)',
    'LDHA': 'Metabolic checkpoint (MIXED attribution)',
    'BAK1': 'Myeloid apoptosis, pan-tissue',
    'STAT1': 'IFN signaling (FDR Tier 1)',
    'IL1RN': 'Tolerance brake',
    'TAP1': 'Antigen processing (FDR Tier 1)',
    'SERPINE1': 'IT-to-IA NK transition',
    'ID3': 'IT-to-IA NK transition',
    'LAYN': 'Liver exhaustion (tissue-opposite)',
    'BCL6': 'Liver CD8T stemness',
    'RORC': 'Liver CD4T Treg skewing',
    'CTLA4': 'Liver CD4T Treg (tissue-opposite)',
    'TIGIT': 'Liver T cell checkpoint',
    'FOXP3': 'IA-AR discriminator',
    'MT-CYB': 'Mito co-regulation hub (rho=0.93)',
    'MEFV': 'Inflammasome sensor, CR scar',
    'B2M': 'Antigen processing',
    'IL1B': 'Inflammasome effector',
    'TET2': 'Epigenetic regulator',
}

# Also load C3b info if available
c3b_genes = set()
try:
    c3b_mask = df_c3_genes['in_C3b'] == True
    c3b_genes = set(df_c3_genes.loc[c3b_mask, 'gene'].tolist())
    c3_strict = set(df_c3_genes.loc[df_c3_genes['in_C3'] == True, 'gene'].tolist())
    print(f"  C3 strict (in_C3=True): {len(c3_strict)}")
    print(f"  C3b (in_C3b=True): {len(c3b_genes)}")
    print(f"  C3b-only (not in C3 strict): {len(c3b_genes - c3_strict)}")
except:
    c3_strict = c3_set
    print("  ⚠️ No C3b column info, using flat C3 list")

# Full gene universe = C3 ∪ C5
all_genes = sorted(c3_set | c5_set)
print(f"  Total gene universe (C3∪C5): {len(all_genes)}")

wb2 = openpyxl.Workbook()
ws2 = wb2.active
ws2.title = "S2_Gene_Candidates"

# Title
ws2.cell(row=1, column=1, value="Supplementary Table S2. Individual Gene Candidates with Analytical Component Source Attribution")
ws2.cell(row=1, column=1).font = title_font
ws2.merge_cells('A1:I1')

ws2.cell(row=2, column=1, value=f"C3: {len(c3_set)} genes (broad screen); C5: {len(c5_set)} genes (refined panel); C9B: {len(C9B_GENES)} genes (re-verified with stratified FDR). {len(shared)} genes shared between C3 and C5.")
ws2.cell(row=2, column=1).font = Font(name='Arial', size=9, italic=True)
ws2.merge_cells('A2:I2')

# Headers
headers2 = ['Gene', 'C3 (196)', 'C3b', 'C5 (148)', 'C9B (19)', 'Source Category', 'Pathway Membership', 'N Pathways', 'Key Manuscript Role']
for i, h in enumerate(headers2, 1):
    ws2.cell(row=4, column=i, value=h)
style_header(ws2, 4, 9)

row = 5
for gene in all_genes:
    in_c3 = gene in c3_set
    in_c5 = gene in c5_set
    in_c9b = gene in C9B_GENES
    in_c3b = gene in c3b_genes

    # Source category
    if in_c3 and in_c5:
        src = "C3+C5 (shared)"
    elif in_c3 and not in_c5:
        src = "C3-only"
    elif in_c5 and not in_c3:
        src = "C5-only"
    else:
        src = "Other"

    # Pathway membership
    pw_list = []
    for pw, genes in ALL_29.items():
        if gene in genes:
            pw_list.append(pw)
    pw_str = ', '.join(pw_list) if pw_list else 'None (literature-derived)'

    ws2.cell(row=row, column=1, value=gene)
    ws2.cell(row=row, column=2, value="Yes" if in_c3 else "No")
    ws2.cell(row=row, column=3, value="Yes" if in_c3b else "")
    ws2.cell(row=row, column=4, value="Yes" if in_c5 else "No")
    ws2.cell(row=row, column=5, value="Yes" if in_c9b else "")
    ws2.cell(row=row, column=6, value=src)
    ws2.cell(row=row, column=7, value=pw_str)
    ws2.cell(row=row, column=8, value=len(pw_list) if pw_list else 0)
    ws2.cell(row=row, column=9, value=KEY_ROLES.get(gene, ''))

    # Color coding
    fill = None
    if in_c9b:
        fill = green_fill
    elif src == "C3-only":
        fill = orange_fill
    style_data(ws2, row, 9, fill=fill)

    for c in [2, 3, 4, 5, 8]:
        ws2.cell(row=row, column=c).alignment = Alignment(horizontal='center')
    row += 1

# Summary rows
row += 1
summaries = [
    ("C3 total", len(c3_set)),
    ("C5 total", len(c5_set)),
    ("C3 ∩ C5 shared", len(shared)),
    ("C3-only", len(c3_set - c5_set)),
    ("C5-only", len(c5_set - c3_set)),
    ("C9B re-verified", len(C9B_GENES)),
    ("Gene universe (C3∪C5)", len(all_genes)),
]
for label, val in summaries:
    ws2.cell(row=row, column=1, value=label)
    ws2.cell(row=row, column=2, value=str(val))
    ws2.cell(row=row, column=1).font = Font(name='Arial', bold=True, size=10)
    ws2.cell(row=row, column=2).font = data_font
    for c in range(1, 10):
        ws2.cell(row=row, column=c).border = thin_border
    row += 1

# Column widths
ws2.column_dimensions['A'].width = 14
ws2.column_dimensions['B'].width = 10
ws2.column_dimensions['C'].width = 8
ws2.column_dimensions['D'].width = 10
ws2.column_dimensions['E'].width = 10
ws2.column_dimensions['F'].width = 18
ws2.column_dimensions['G'].width = 55
ws2.column_dimensions['H'].width = 12
ws2.column_dimensions['I'].width = 42

# Legend sheet
ws_leg = wb2.create_sheet("Legend")
ws_leg.cell(row=1, column=1, value="Color Legend & Definitions")
ws_leg.cell(row=1, column=1).font = Font(name='Arial', bold=True, size=11)
legends = [
    ("Green", "C9B re-verified gene (stratified FDR applied)", 'E2EFDA'),
    ("Orange", "C3-only gene (not in C5 148-gene panel)", 'FCE4D6'),
    ("No color", "C3+C5 shared or C5-only gene", None),
]
for i, (label, desc, color) in enumerate(legends, 3):
    ws_leg.cell(row=i, column=1, value=label)
    ws_leg.cell(row=i, column=2, value=desc)
    if color:
        ws_leg.cell(row=i, column=1).fill = PatternFill('solid', fgColor=color)

ws_leg.cell(row=7, column=1, value="Column Definitions")
ws_leg.cell(row=7, column=1).font = Font(name='Arial', bold=True, size=11)
defs = [
    ("C3 (196)", "Broad individual gene expression screen (196 candidates)"),
    ("C3b", "Genes added via C3b supplementary analysis (e.g., AICDA, JCHAIN)"),
    ("C5 (148)", "Refined gene panel after expression filtering"),
    ("C9B (19)", "Critical C3-only genes re-verified with stratified FDR in C9B"),
    ("Source Category", "C3+C5 shared / C3-only / C5-only"),
    ("Pathway Membership", "Which of the 29 pathways this gene belongs to (if any)"),
    ("Key Manuscript Role", "Role of this gene in manuscript findings (blank = not a key finding)"),
]
for i, (col, desc) in enumerate(defs, 8):
    ws_leg.cell(row=i, column=1, value=col)
    ws_leg.cell(row=i, column=2, value=desc)
    ws_leg.cell(row=i, column=1).font = Font(name='Arial', bold=True, size=10)

ws_leg.column_dimensions['A'].width = 22
ws_leg.column_dimensions['B'].width = 70

s2_path = f'{OUT_DIR}/Supplementary_Table_S2.xlsx'
wb2.save(s2_path)
print(f"  ✅ S2 saved: {s2_path}")
print(f"     {len(all_genes)} genes, C3={len(c3_set)}, C5={len(c5_set)}, Shared={len(shared)}")

In [ ]:
# =================================================================
# CELL 11: FINAL VERIFICATION OF GENERATED FILES
# =================================================================
print("=" * 70)
print("  FINAL VERIFICATION OF GENERATED S1/S2")
print("=" * 70)

# Re-read and verify
wb1_check = openpyxl.load_workbook(s1_path)
ws1_check = wb1_check.active
pw_count = 0
for r in range(5, ws1_check.max_row):
    if ws1_check.cell(r, 2).value and ws1_check.cell(r, 1).value != "TOTAL":
        pw_count += 1
print(f"  S1: {pw_count} pathways listed")

wb2_check = openpyxl.load_workbook(s2_path)
ws2_check = wb2_check.active
c3_yes = sum(1 for r in range(5, ws2_check.max_row) if ws2_check.cell(r, 2).value == 'Yes')
c5_yes = sum(1 for r in range(5, ws2_check.max_row) if ws2_check.cell(r, 4).value == 'Yes')
c9b_yes = sum(1 for r in range(5, ws2_check.max_row) if ws2_check.cell(r, 5).value == 'Yes')
print(f"  S2: C3=Yes:{c3_yes}, C5=Yes:{c5_yes}, C9B=Yes:{c9b_yes}")

# Cross-check against manuscript numbers
print(f"\n  === MANUSCRIPT NUMBER VERIFICATION ===")
checks = [
    ("29 pathways (S1)", pw_count, 29),
    (f"Unique pathway genes", n_unique, 179),
    ("C3 genes (S2)", c3_yes, 196),
    ("C5 genes (S2)", c5_yes, 148),
    ("C3∩C5 shared", len(shared), 81),
    ("C9B genes", c9b_yes, 19),
]
all_pass = True
for label, actual, expected in checks:
    ok = actual == expected
    if not ok:
        all_pass = False
    print(f"  {'✅' if ok else '❌'} {label}: {actual} (expected {expected})")

print(f"\n  === NUMBERS TO UPDATE IN MANUSCRIPT M&M ===")
print(f"  '182 unique genes' → {n_unique}")
print(f"  '102 genes shared' → {len(shared)}")
print(f"  All other numbers (196, 148, 29, 19) → ✅ confirmed correct")

if all_pass:
    print(f"\n  🎉 ALL CHECKS PASSED — S1/S2 are publication-ready")
else:
    print(f"\n  ⚠️ Some checks failed — review above")

print(f"\n  Files saved:")
print(f"    {s1_path}")
print(f"    {s2_path}")
print(f"\n{'=' * 70}")
print(f"  ✅ S1/S2 GENERATION COMPLETE")
print(f"{'=' * 70}")